# Day 14 — Pandas Fundamentals
## Production OEE Analysis with Pandas

Loading and analyzing 240 production records using Pandas DataFrame operations.

In [1]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("sample_data.csv")
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

Dataset loaded: 240 rows × 8 columns
Columns: ['date', 'machine', 'shift', 'actual_run', 'planned', 'units', 'good_units', 'defect_code']


In [2]:
print(df.head())
print()
print(df.dtypes)

         date          machine  shift  actual_run  planned  units  good_units  \
0  2024-01-01      CNC Mill #4    Day         461      480    707         707   
1  2024-01-01      CNC Mill #4  Night         411      480    764         760   
2  2024-01-01         Press #2    Day         466      480    694         676   
3  2024-01-01         Press #2  Night         384      480    665         663   
4  2024-01-01  Assembly Line A    Day         409      480    908         908   

  defect_code  
0     DEF-007  
1     DEF-001  
2     DEF-011  
3     DEF-004  
4        NONE  

date           object
machine        object
shift          object
actual_run      int64
planned         int64
units           int64
good_units      int64
defect_code    object
dtype: object


In [3]:
df["availability"] = df["actual_run"] / df["planned"]
df["performance"]  = df["units"] / (df["planned"] * 1.8)
df["quality"]      = df["good_units"] / df["units"]
df["oee"]          = df["availability"] * df["performance"] * df["quality"]

print(f"OEE columns added. New shape: {df.shape}")
print(df[["machine", "shift", "oee"]].head(8))

OEE columns added. New shape: (240, 12)
           machine  shift       oee
0      CNC Mill #4    Day  0.785897
1      CNC Mill #4  Night  0.753183
2         Press #2    Day  0.759587
3         Press #2  Night  0.613889
4  Assembly Line A    Day  0.895476
5  Assembly Line A  Night  0.836914
6   Weld Station B    Day  0.868924
7   Weld Station B  Night  0.968036


## Fleet OEE Summary

Average OEE grouped by machine center across all shifts and dates.

In [4]:
summary = df.groupby("machine").agg(
    avg_oee      = ("oee", "mean"),
    min_oee      = ("oee", "min"),
    max_oee      = ("oee", "max"),
    record_count = ("oee", "count")
).round(3)

print(summary)

                 avg_oee  min_oee  max_oee  record_count
machine                                                 
Assembly Line A    0.809    0.614    1.015            60
CNC Mill #4        0.805    0.634    1.046            60
Press #2           0.802    0.614    1.023            60
Weld Station B     0.806    0.605    1.019            60


In [5]:
shift_summary = df.groupby(["machine", "shift"])["oee"].mean().round(3)
print("\nOEE by Machine and Shift:")
print(shift_summary)


OEE by Machine and Shift:
machine          shift
Assembly Line A  Day      0.814
                 Night    0.804
CNC Mill #4      Day      0.788
                 Night    0.822
Press #2         Day      0.806
                 Night    0.798
Weld Station B   Day      0.791
                 Night    0.821
Name: oee, dtype: float64


In [6]:
below = df[df["oee"] < 0.70][["date", "machine", "shift", "oee"]]
print(f"\nRecords below OEE threshold: {len(below)}")
print(below.head(10))


Records below OEE threshold: 34
          date          machine  shift       oee
3   2024-01-01         Press #2  Night  0.613889
13  2024-01-02  Assembly Line A  Night  0.666319
21  2024-01-03  Assembly Line A  Night  0.646952
22  2024-01-03   Weld Station B    Day  0.675825
29  2024-01-04  Assembly Line A  Night  0.690784
32  2024-01-05      CNC Mill #4    Day  0.634816
51  2024-01-07         Press #2  Night  0.627652
52  2024-01-07  Assembly Line A    Day  0.613735
54  2024-01-07   Weld Station B    Day  0.685988
56  2024-01-08      CNC Mill #4    Day  0.677035


## Key Findings

- Fleet OEE calculated across 240 records
- Grouped by machine center and shift
- Records flagged below 0.70 threshold for investigation

*Next session: Matplotlib visualization*